In [ ]:
# Setup: imports and reproducibility
import os
from pathlib import Path
from typing import Any, cast

import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TRACKING_URI = os.getenv(
    "MLFLOW_TRACKING_URI",
    "https://dagshub.com/PetelinekBenjamin/air-quality-comparison.mlflow",
)
EXPERIMENTS = {
    "standard": "comperosion-normal",
    "five_fold": "comperosion-5fold",
}
SUMMARY_METRICS = ["mae", "rmse", "mape", "smape", "r2", "wape", "mse"]
SLOT_KEYS = ["tags.data_source", "tags.scenario", "tags.fold", "tags.model", "tags.approach"]

mlflow.set_tracking_uri(TRACKING_URI)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
plt.style.use("ggplot")

print(f"Tracking URI: {TRACKING_URI}")
print("Eksperimenti:", EXPERIMENTS)



1. Preberemo rune iz eksperimentov `comperosion-normal` in `comperosion-5fold`.
2. Obdrzimo samo `FINISHED` run-e z imenom `forecast-*` (brez summary run-ov).
3. Ker `normal` vsebuje stare ponovitve, za vsak slot (`data_source/scenario/fold/model/approach`) obdrzimo samo **najnovejsi** run.
4. Izrisemo primerjavo metrik in shranimo tabele v `reports/forecast_experiments/mlflow_latest_analysis/`.


In [ ]:
# Define parameters and lightweight helpers

def _ensure_column(df: pd.DataFrame, column: str, default_value: Any) -> None:
    if column not in df.columns:
        df[column] = default_value


def _search_runs_df(experiment_name: str, max_results: int) -> pd.DataFrame:
    # Pylance-friendly: mlflow stubs may expose list[Run], runtime returns DataFrame.
    raw_runs = mlflow.search_runs(
        experiment_names=[experiment_name],
        max_results=max_results,
        order_by=["attributes.start_time DESC"],
    )
    if isinstance(raw_runs, pd.DataFrame):
        return raw_runs.copy()
    return pd.DataFrame(cast(Any, raw_runs)).copy()


def load_experiment_runs(experiment_name: str, scenario_label: str, max_results: int = 5000) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = _search_runs_df(experiment_name, max_results)
    if df.empty:
        return df, df

    _ensure_column(df, "status", "")
    _ensure_column(df, "tags.mlflow.runName", "")

    df = df.loc[df["status"] == "FINISHED"].copy()
    df = df.loc[df["tags.mlflow.runName"].astype(str).str.startswith("forecast-", na=False)].copy()
    if df.empty:
        return df, df

    for col in SLOT_KEYS:
        _ensure_column(df, col, "")
    _ensure_column(df, "start_time", pd.NaT)
    _ensure_column(df, "end_time", pd.NaT)
    if "run_id" not in df.columns:
        df["run_id"] = df.index.astype(str)

    df["start_time"] = pd.to_datetime(df["start_time"], utc=True, errors="coerce")
    df["end_time"] = pd.to_datetime(df["end_time"], utc=True, errors="coerce")
    df["duration_min"] = (df["end_time"] - df["start_time"]).dt.total_seconds().div(60.0)

    df["scenario"] = scenario_label
    df["data_source"] = df["tags.data_source"].astype(str)
    df["model"] = df["tags.model"].astype(str)
    df["approach"] = df["tags.approach"].astype(str)
    df["model_approach"] = df["model"] + "__" + df["approach"]
    df["fold"] = pd.to_numeric(df["tags.fold"], errors="coerce").fillna(0).astype(int)

    for metric in SUMMARY_METRICS:
        metric_col = f"metrics.summary.{metric}"
        if metric_col in df.columns:
            df[metric] = pd.to_numeric(df[metric_col], errors="coerce")
        else:
            df[metric] = np.nan

    latest = (
        df.sort_values(["start_time", "run_id"], ascending=[False, False])
        .drop_duplicates(subset=SLOT_KEYS, keep="first")
        .reset_index(drop=True)
    )

    return df.reset_index(drop=True), latest


raw_frames: list[pd.DataFrame] = []
latest_frames: list[pd.DataFrame] = []
for scenario_name, experiment_name in EXPERIMENTS.items():
    raw_df, latest_df = load_experiment_runs(experiment_name, scenario_name)
    print(
        f"{scenario_name:>9} | experiment={experiment_name} | "
        f"all_finished={len(raw_df):3d} | latest_slots={len(latest_df):3d}"
    )
    if not raw_df.empty:
        raw_frames.append(raw_df)
    if not latest_df.empty:
        latest_frames.append(latest_df)

latest_all = pd.concat(latest_frames, ignore_index=True) if latest_frames else pd.DataFrame()
if latest_all.empty:
    raise ValueError("Ni najdenih run-ov za analizo. Preveri tracking URI in pravice dostopa.")

agg = (
    latest_all.groupby(["scenario", "model", "approach", "model_approach"], as_index=False)
    .agg(
        n_runs=("run_id", "count"),
        mean_rmse=("rmse", "mean"),
        std_rmse=("rmse", "std"),
        mean_mae=("mae", "mean"),
        mean_mape=("mape", "mean"),
        mean_smape=("smape", "mean"),
        mean_wape=("wape", "mean"),
        mean_r2=("r2", "mean"),
    )
)



Primerjava `latest` runov za oba scenarija (`standard`, `five_fold`) in osnovni izris grafov.


In [ ]:
# Record findings in a minimal, copy-pasteable structure
view_cols = [
    "scenario",
    "fold",
    "data_source",
    "model",
    "approach",
    "run_id",
    "start_time",
    "rmse",
    "mae",
    "mape",
    "smape",
    "wape",
    "r2",
    "duration_min",
]

latest_view = latest_all[view_cols].sort_values(["scenario", "fold", "model", "approach"]).reset_index(drop=True)
print("Latest runi (po slotih):", len(latest_view))
print(latest_view.head(50).to_string(index=False))

agg_sorted = agg.sort_values(["scenario", "mean_rmse", "mean_mae", "mean_smape", "mean_wape"]).reset_index(drop=True)
print("\nPovprecne metrike po model+approach:")
print(agg_sorted.to_string(index=False))

fig, axes = plt.subplots(1, 5, figsize=(32, 5), constrained_layout=True)
for ax, metric, title, ascending in [
    (axes[0], "mean_rmse", "Mean RMSE (lower is better)", True),
    (axes[1], "mean_mae", "Mean MAE (lower is better)", True),
    (axes[2], "mean_smape", "Mean sMAPE (lower is better)", True),
    (axes[3], "mean_wape", "Mean WAPE (lower is better)", True),
    (axes[4], "mean_r2", "Mean R2 (higher is better)", False),
]:
    pivot = agg.pivot(index="model_approach", columns="scenario", values=metric)
    if pivot.empty:
        continue
    sort_col = "standard" if "standard" in pivot.columns else pivot.columns[0]
    pivot = pivot.sort_values(sort_col, ascending=ascending)
    pivot.plot(kind="bar", ax=ax, width=0.82)
    ax.set_title(title)
    ax.set_xlabel("model__approach")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=60)
    ax.grid(axis="y", alpha=0.35)

plt.show()

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
plot_df = agg_sorted.copy().sort_values(["mean_rmse", "mean_r2", "model_approach"]).reset_index(drop=True)


def _label_offsets(df: pd.DataFrame, x_col: str, y_col: str) -> pd.DataFrame:
    work = df.copy()
    x_span = max(float(work[x_col].max() - work[x_col].min()), 1e-9)
    y_span = max(float(work[y_col].max() - work[y_col].min()), 1e-9)
    bin_w_x = max(x_span * 0.04, 1e-9)
    bin_w_y = max(y_span * 0.04, 1e-9)

    work["_x_bin"] = np.floor((work[x_col] - work[x_col].min()) / bin_w_x).astype(int)
    work["_y_bin"] = np.floor((work[y_col] - work[y_col].min()) / bin_w_y).astype(int)
    work["_label_slot"] = work.groupby(["_x_bin", "_y_bin"]).cumcount()

    base_offsets = [
        (8, 8),
        (8, -8),
        (-8, 8),
        (-8, -8),
        (12, 0),
        (0, 12),
        (-12, 0),
        (0, -12),
        (14, 10),
        (-14, 10),
        (14, -10),
        (-14, -10),
    ]

    offset_x: list[int] = []
    offset_y: list[int] = []
    for slot in work["_label_slot"]:
        dx, dy = base_offsets[int(slot) % len(base_offsets)]
        scale = 1 + int(slot) // len(base_offsets)
        offset_x.append(dx * scale)
        offset_y.append(dy * scale)

    work["label_dx"] = offset_x
    work["label_dy"] = offset_y
    return work.drop(columns=["_x_bin", "_y_bin", "_label_slot"])


plot_df = _label_offsets(plot_df, "mean_rmse", "mean_r2")

for scenario in sorted(plot_df["scenario"].unique()):
    sub = plot_df[plot_df["scenario"] == scenario]
    ax.scatter(
        sub["mean_rmse"],
        sub["mean_r2"],
        s=120,
        alpha=0.85,
        label=scenario,
    )
    for _, row in sub.iterrows():
        label = row["model"] + "__" + row["approach"]
        ax.annotate(
            label,
            (row["mean_rmse"], row["mean_r2"]),
            xytext=(int(row["label_dx"]), int(row["label_dy"])),
            textcoords="offset points",
            fontsize=8,
            alpha=0.9,
            bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "none", "alpha": 0.7},
            arrowprops={"arrowstyle": "-", "color": "#666666", "lw": 0.6, "alpha": 0.6},
        )

ax.set_title("Model trade-off: RMSE vs R2")
ax.set_xlabel("mean_rmse (lower better)")
ax.set_ylabel("mean_r2 (higher better)")
ax.grid(alpha=0.3)
ax.legend(title="scenario")
plt.show()

scenario_means = agg.groupby(["scenario", "model_approach"], as_index=False).agg(
    mean_rmse=("mean_rmse", "mean"),
    mean_mae=("mean_mae", "mean"),
    mean_smape=("mean_smape", "mean"),
    mean_wape=("mean_wape", "mean"),
    mean_r2=("mean_r2", "mean"),
)

rmse_p = scenario_means.pivot(index="model_approach", columns="scenario", values="mean_rmse")
mae_p = scenario_means.pivot(index="model_approach", columns="scenario", values="mean_mae")
smape_p = scenario_means.pivot(index="model_approach", columns="scenario", values="mean_smape")
wape_p = scenario_means.pivot(index="model_approach", columns="scenario", values="mean_wape")
r2_p = scenario_means.pivot(index="model_approach", columns="scenario", values="mean_r2")

if {"standard", "five_fold"}.issubset(set(scenario_means["scenario"].unique())):
    delta = pd.DataFrame(index=sorted(set(rmse_p.index) | set(mae_p.index) | set(smape_p.index) | set(wape_p.index) | set(r2_p.index)))
    std_rmse = rmse_p["standard"] if "standard" in rmse_p.columns else pd.Series(index=delta.index, dtype=float)
    ff_rmse = rmse_p["five_fold"] if "five_fold" in rmse_p.columns else pd.Series(index=delta.index, dtype=float)
    std_mae = mae_p["standard"] if "standard" in mae_p.columns else pd.Series(index=delta.index, dtype=float)
    ff_mae = mae_p["five_fold"] if "five_fold" in mae_p.columns else pd.Series(index=delta.index, dtype=float)
    std_smape = smape_p["standard"] if "standard" in smape_p.columns else pd.Series(index=delta.index, dtype=float)
    ff_smape = smape_p["five_fold"] if "five_fold" in smape_p.columns else pd.Series(index=delta.index, dtype=float)
    std_wape = wape_p["standard"] if "standard" in wape_p.columns else pd.Series(index=delta.index, dtype=float)
    ff_wape = wape_p["five_fold"] if "five_fold" in wape_p.columns else pd.Series(index=delta.index, dtype=float)
    std_r2 = r2_p["standard"] if "standard" in r2_p.columns else pd.Series(index=delta.index, dtype=float)
    ff_r2 = r2_p["five_fold"] if "five_fold" in r2_p.columns else pd.Series(index=delta.index, dtype=float)

    delta["delta_rmse_5fold_minus_standard"] = ff_rmse.reindex(delta.index) - std_rmse.reindex(delta.index)
    delta["delta_mae_5fold_minus_standard"] = ff_mae.reindex(delta.index) - std_mae.reindex(delta.index)
    delta["delta_smape_5fold_minus_standard"] = ff_smape.reindex(delta.index) - std_smape.reindex(delta.index)
    delta["delta_wape_5fold_minus_standard"] = ff_wape.reindex(delta.index) - std_wape.reindex(delta.index)
    delta["delta_r2_5fold_minus_standard"] = ff_r2.reindex(delta.index) - std_r2.reindex(delta.index)
    delta = delta.sort_values("delta_rmse_5fold_minus_standard")
    print("\nDelta (five_fold - standard):")
    print(delta.to_string())

    fig, axes = plt.subplots(2, 2, figsize=(18, 10), constrained_layout=True)
    axes = axes.ravel()

    delta[["delta_rmse_5fold_minus_standard"]].plot(kind="bar", ax=axes[0], legend=False, color="#d95f02")
    axes[0].axhline(0.0, color="black", linewidth=1)
    axes[0].set_title("Delta RMSE: five_fold - standard")
    axes[0].set_xlabel("model__approach")
    axes[0].set_ylabel("delta_rmse")
    axes[0].tick_params(axis="x", rotation=60)
    axes[0].grid(axis="y", alpha=0.35)

    delta[["delta_smape_5fold_minus_standard"]].plot(kind="bar", ax=axes[1], legend=False, color="#7570b3")
    axes[1].axhline(0.0, color="black", linewidth=1)
    axes[1].set_title("Delta sMAPE: five_fold - standard")
    axes[1].set_xlabel("model__approach")
    axes[1].set_ylabel("delta_smape")
    axes[1].tick_params(axis="x", rotation=60)
    axes[1].grid(axis="y", alpha=0.35)

    delta[["delta_wape_5fold_minus_standard"]].plot(kind="bar", ax=axes[2], legend=False, color="#e7298a")
    axes[2].axhline(0.0, color="black", linewidth=1)
    axes[2].set_title("Delta WAPE: five_fold - standard")
    axes[2].set_xlabel("model__approach")
    axes[2].set_ylabel("delta_wape")
    axes[2].tick_params(axis="x", rotation=60)
    axes[2].grid(axis="y", alpha=0.35)

    delta[["delta_r2_5fold_minus_standard"]].plot(kind="bar", ax=axes[3], legend=False, color="#1b9e77")
    axes[3].axhline(0.0, color="black", linewidth=1)
    axes[3].set_title("Delta R2: five_fold - standard")
    axes[3].set_xlabel("model__approach")
    axes[3].set_ylabel("delta_r2")
    axes[3].tick_params(axis="x", rotation=60)
    axes[3].grid(axis="y", alpha=0.35)

    plt.show()
else:
    delta = pd.DataFrame()
    print("\nDelta ni izracunan, ker manjka eden od scenarijev.")

output_dir = Path("reports/forecast_experiments/mlflow_latest_analysis")
output_dir.mkdir(parents=True, exist_ok=True)
latest_view.to_csv(output_dir / "latest_runs_table.csv", index=False)
agg_sorted.to_csv(output_dir / "latest_runs_aggregated_metrics.csv", index=False)
if not delta.empty:
    delta.reset_index(names=["model_approach"]).to_csv(output_dir / "latest_runs_delta_five_fold_vs_standard.csv", index=False)

print(f"\nSaved tables to: {output_dir}")
